# Single-cell perturbations: sparse-cell validation workflow

Generate four sparse-cell folds and automatically evaluate every baseline on each fold.


In [46]:
%pip install kagglehub pandas pyarrow
%run download.py

from pathlib import Path

PROJECT_DIR = Path.cwd()
DATA_DIR = Path(path)

Note: you may need to restart the kernel to use updated packages.
Path to competition files: C:\Users\73714\.cache\kagglehub\competitions\open-problems-single-cell-perturbations


## 1. Load labeled DE data

Each row contains one cell-type–drug pair and its gene-level DE target values.

In [47]:
import pandas as pd
from IPython.display import display
from split_data import save_sparse_splits, apply_split, get_gene_columns
from baseline_pca_ridge import fit_predict, mrrmse
from baseline_ridge import fit_predict as fit_predict_ridge

de_train = pd.read_parquet(DATA_DIR / "de_train.parquet")
id_map = pd.read_csv(DATA_DIR / "id_map.csv")
gene_cols = get_gene_columns(de_train)

print(f"Labeled pairs: {len(de_train)}")
print(f"DE target genes: {len(gene_cols)}")
print(f"Official test pairs: {len(id_map)}")
display(de_train.groupby("cell_type").size().rename("available_pairs").to_frame())

Labeled pairs: 614
DE target genes: 18211
Official test pairs: 255


,available_pairs
cell_type,
B cells,17
Myeloid cells,17
NK cells,146
T cells CD4+,146
T cells CD8+,142
T regulatory cells,146


## 2. Generate the four sparse-cell folds

Use the anchor-drug rules from XGBoost.ipynb. Each fold holds out non-anchor, non-control pairs from one source cell type.


In [48]:
split_manifests = save_sparse_splits(de_train, PROJECT_DIR / "splits")
folds = []
for target_cell, manifest in split_manifests.items():
    train_fold, val_fold = apply_split(de_train, manifest)
    folds.append((target_cell, train_fold, val_fold))

display(pd.DataFrame([
    {"target_cell": target_cell, "train": len(train_fold),
     "target_anchor_drugs": int((train_fold.cell_type.eq(target_cell) & ~train_fold.control).sum()),
     "validation": len(val_fold)}
    for target_cell, train_fold, val_fold in folds
]))


,target_cell,train,target_anchor_drugs,validation
0,NK cells,485,15,129
1,T cells CD4+,485,15,129
2,T cells CD8+,487,13,127
3,T regulatory cells,485,15,129


## 3. Evaluate each method

In [49]:
import numpy as np
from baseline_pca_lasso import fit_predict as fit_predict_lasso
from baseline_lasso import fit_predict as fit_predict_direct_lasso

feature_cols = ["cell_type", "sm_name"]
X_test = id_map[feature_cols].copy()
mean_scores = {}
validated_settings = {}

def evaluate_method(name, predict, **settings):
    scores = []
    for target_cell, train, validation in folds:
        prediction = predict(train[feature_cols], train[gene_cols],
                             validation[feature_cols], **settings)
        scores.append(mrrmse(validation[gene_cols], prediction))
    mean_scores[name] = np.mean(scores)
    validated_settings[name] = settings.copy()
    print(f"{name}: {mean_scores[name]:.6f}")


### Zero

In [50]:
def predict_zero(x, y, x_val):
    return np.zeros((len(x_val), y.shape[1]))

evaluate_method("Zero", predict_zero)

Zero: 1.005772


### Gene mean

In [51]:
def predict_mean(x, y, x_val):
    return np.tile(y.mean().to_numpy(), (len(x_val), 1))

evaluate_method("Gene mean", predict_mean)

Gene mean: 1.106285


### Ridge

In [52]:
RIDGE_ALPHA_RIDGE = 100000
evaluate_method("Ridge", fit_predict_ridge, alpha=RIDGE_ALPHA_RIDGE)

Ridge: 1.106340


### PCA + Ridge

In [53]:
evaluate_method("PCA + Ridge", fit_predict, n_components=200, alpha=100000)

PCA + Ridge: 1.106340


### Lasso

In [54]:
evaluate_method("Lasso", fit_predict_direct_lasso, alpha=100)

Lasso: 1.106285


### PCA + Lasso

In [55]:
evaluate_method("PCA + Lasso", fit_predict_lasso, n_components=10, alpha=100)

PCA + Lasso: 1.106285


### MRRMSE + L2

In [56]:
from baseline_MRRMSE_L2 import fit_predict as fit_predict_mrrmse_l2

MRRMSE_L2_ALPHA = 0.03
evaluate_method("MRRMSE + L2", fit_predict_mrrmse_l2, alpha=MRRMSE_L2_ALPHA)


MRRMSE + L2: 0.974028


### MRRMSE + L1

In [57]:
from baseline_MRRMSE_L1 import fit_predict as fit_mrrmse_l1

evaluate_method("MRRMSE + L1", fit_mrrmse_l1, alpha=0.01)


MRRMSE + L1: 0.981585


### PCA + MRRMSE + L2

In [58]:
from baseline_PCA_MRRMSE_L2 import fit_predict as fit_pca_mrrmse_l2

evaluate_method("PCA + MRRMSE + L2", fit_pca_mrrmse_l2, n_components=100, alpha=0.01)


PCA + MRRMSE + L2: 0.969221


### PCA + MRRMSE + L1

In [59]:
from baseline_PCA_MRRMSE_L1 import fit_predict as fit_pca_mrrmse_l1

evaluate_method("PCA + MRRMSE + L1", fit_pca_mrrmse_l1, n_components=100, alpha=1)


PCA + MRRMSE + L1: 0.998887


### Four-fold summary

In [60]:
display(pd.Series(mean_scores, name="four_fold_mean_MRRMSE").rename_axis("model").sort_values().to_frame())

,four_fold_mean_MRRMSE
model,
PCA + MRRMSE + L2,0.969221
MRRMSE + L2,0.974028
MRRMSE + L1,0.981585
PCA + MRRMSE + L1,0.998887
Zero,1.005772
PCA + Lasso,1.106285
Lasso,1.106285
Gene mean,1.106285
Ridge,1.106340


### MRRMSE constant vs. MRRMSE + L2

In [61]:
def fit_mrrmse_constant(y_train):
    # The geometric median minimizes mean row RMSE in gene space.
    y = y_train.to_numpy(dtype=float)
    center = y.mean(axis=0)
    for _ in range(300):
        row_rmse = np.sqrt(np.mean((y - center) ** 2, axis=1))
        weights = 1 / np.maximum(row_rmse, 1e-10)
        updated = np.average(y, axis=0, weights=weights)
        if np.sqrt(np.mean((updated - center) ** 2)) < 1e-8:
            return updated
        center = updated
    raise RuntimeError("The constant fit did not converge.")

constant_scores = []
for _, train, validation in folds:
    center = fit_mrrmse_constant(train[gene_cols])
    truth = validation[gene_cols]
    prediction = np.broadcast_to(center, truth.shape)
    constant_scores.append(mrrmse(truth, prediction))

display(pd.Series({
    "MRRMSE constant": np.mean(constant_scores),
    "MRRMSE + L2": mean_scores["MRRMSE + L2"],
}, name="four_fold_mean_MRRMSE").to_frame())


,four_fold_mean_MRRMSE
MRRMSE constant,0.999504
MRRMSE + L2,0.974028


## 4. Official test predictions

Each method is refitted on all labeled pairs using its recorded validation settings. Run each cell to save its CSV in submissions/.


In [62]:
# Run validation cells first to record the parameters used for each model.
sample = pd.read_csv(DATA_DIR / "sample_submission.csv")
assert sample["id"].is_unique and id_map["id"].is_unique
assert set(sample["id"]) == set(id_map["id"])
submission_genes = sample.columns.drop("id").tolist()
assert set(submission_genes) == set(gene_cols)
final_X_train = de_train[feature_cols].copy()
final_y_train = de_train[gene_cols].copy()
final_X_test = id_map.set_index("id").loc[sample["id"], feature_cols].reset_index(drop=True)
SUBMISSION_DIR = PROJECT_DIR / "submissions"
SUBMISSION_DIR.mkdir(exist_ok=True)

def save_submission(name, filename, predict):
    predictions = predict(final_X_train, final_y_train, final_X_test,
                          **validated_settings[name])
    if isinstance(predictions, pd.DataFrame):
        assert predictions.columns.equals(final_y_train.columns)
    values = np.asarray(predictions, dtype=float)
    assert values.shape == (len(sample), len(gene_cols))
    assert np.isfinite(values).all()
    submission = pd.DataFrame(values, columns=gene_cols)
    submission = submission[submission_genes]
    submission.insert(0, "id", sample["id"].to_numpy())
    assert submission.columns.tolist() == sample.columns.tolist()
    output = SUBMISSION_DIR / filename
    temporary = output.with_suffix(".csv.tmp")
    submission.to_csv(temporary, index=False)
    temporary.replace(output)
    print(f"{name}: saved {output.name}, shape {submission.shape}")


### Zero

In [63]:
save_submission("Zero", "submission_zero.csv", predict_zero)


Zero: saved submission_zero.csv, shape (255, 18212)


### Gene mean

In [64]:
save_submission("Gene mean", "submission_gene_mean.csv", predict_mean)


Gene mean: saved submission_gene_mean.csv, shape (255, 18212)


### Ridge

In [65]:
save_submission("Ridge", "submission_ridge.csv", fit_predict_ridge)


Ridge: saved submission_ridge.csv, shape (255, 18212)


### PCA + Ridge

In [66]:
save_submission("PCA + Ridge", "submission_pca_ridge.csv", fit_predict)


PCA + Ridge: saved submission_pca_ridge.csv, shape (255, 18212)


### Lasso

In [67]:
save_submission("Lasso", "submission_lasso.csv", fit_predict_direct_lasso)


Lasso: saved submission_lasso.csv, shape (255, 18212)


### PCA + Lasso

In [68]:
save_submission("PCA + Lasso", "submission_pca_lasso.csv", fit_predict_lasso)


PCA + Lasso: saved submission_pca_lasso.csv, shape (255, 18212)


### MRRMSE + L2

In [69]:
save_submission("MRRMSE + L2", "submission_mrrmse_l2.csv", fit_predict_mrrmse_l2)


MRRMSE + L2: saved submission_mrrmse_l2.csv, shape (255, 18212)


### MRRMSE + L1

In [70]:
save_submission("MRRMSE + L1", "submission_mrrmse_l1.csv", fit_mrrmse_l1)


MRRMSE + L1: saved submission_mrrmse_l1.csv, shape (255, 18212)


### PCA + MRRMSE + L2

In [71]:
save_submission("PCA + MRRMSE + L2", "submission_pca_mrrmse_l2.csv", fit_pca_mrrmse_l2)


PCA + MRRMSE + L2: saved submission_pca_mrrmse_l2.csv, shape (255, 18212)


### PCA + MRRMSE + L1

In [72]:
save_submission("PCA + MRRMSE + L1", "submission_pca_mrrmse_l1.csv", fit_pca_mrrmse_l1)


PCA + MRRMSE + L1: saved submission_pca_mrrmse_l1.csv, shape (255, 18212)


### MRRMSE constant

In [73]:
def predict_mrrmse_constant(x, y, x_test):
    center = fit_mrrmse_constant(y)
    return np.broadcast_to(center, (len(x_test), len(center)))

validated_settings["MRRMSE constant"] = {}
save_submission("MRRMSE constant", "submission_mrrmse_constant.csv", predict_mrrmse_constant)


MRRMSE constant: saved submission_mrrmse_constant.csv, shape (255, 18212)
